# Fine-tune Veronica (free Colab GPU)

This notebook teaches a **1B open-weight model** to be Veronica, gophergeist's search ghost: answer only from the
numbered sources, cite them like `[1]`, keep it short, and say "the archives are silent" when the sources don't
cover the question. A small model that already knows the job answers well from a short prompt, and that is what
makes it fast on a CPU.

1. A bigger open **teacher** (`qwen2.5:7b`, Apache 2.0) runs in Ollama right here and writes practice questions
   and model answers from your gopherhole.
2. **Unsloth** LoRA-fine-tunes `Llama-3.2-1B-Instruct` on them.
3. The result is exported to a 4-bit **GGUF** you download and load into Ollama on your own machine.

**Runtime → Change runtime type → T4 GPU** before you start. Expect roughly 20–40 minutes end to end.
The facts still come from retrieval at runtime, so editing your files never needs retraining.

## 1. Install gophergeist, Ollama and Unsloth

In [ ]:
# Point this at your copy of gophergeist (or upload the folder with the Files panel and skip the clone).
REPO_URL = "https://github.com/YOUR-NAME/gophergeist"
HOLE = "gophergeist/demo/hole"   # the folder(s) to learn from; your own text folders work too

!git clone -q {REPO_URL} gophergeist || echo "clone failed: upload the folder instead"
!pip install -q -e gophergeist
!pip install -q unsloth
!curl -fsSL https://ollama.com/install.sh | sh

## 2. Let the teacher write Veronica's lessons

In [ ]:
import subprocess, time
ollama = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5:7b
!gophergeist dataset {HOLE} --model qwen2.5:7b --per-passage 8 -o veronica-train.jsonl --eval veronica-eval.jsonl
!head -c 1500 veronica-train.jsonl

In [ ]:
# Free the GPU for training.
!ollama stop qwen2.5:7b
ollama.terminate()

## 3. Fine-tune Llama 3.2 1B with LoRA

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Llama-3.2-1B-Instruct", max_seq_length=2048, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=7,
)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False) for m in batch["messages"]]}

data = load_dataset("json", data_files={"train": "veronica-train.jsonl", "eval": "veronica-eval.jsonl"})
data = data.map(to_text, batched=True)
print(data)

In [ ]:
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=data["train"], eval_dataset=data["eval"],
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=2048,
        per_device_train_batch_size=4, gradient_accumulation_steps=2,
        num_train_epochs=3, learning_rate=2e-4, warmup_steps=5,
        lr_scheduler_type="linear", logging_steps=5, eval_strategy="epoch",
        optim="adamw_8bit", seed=7, output_dir="outputs", report_to="none",
    ),
)
# Only learn Veronica's replies, not to reproduce the sources and questions.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)
trainer.train()

## 4. Ask the student a few held-out questions

In [ ]:
FastLanguageModel.for_inference(model)
for row in data["eval"].select(range(min(4, len(data["eval"])))):
    msgs = row["messages"][:2]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=160, temperature=0.3, do_sample=True)
    question = msgs[1]["content"].rsplit("Question: ", 1)[-1]
    print("Q:", question)
    print("Student:", tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip())
    print("Teacher:", row["messages"][2]["content"], "\n")

## 5. Export to GGUF and download

In [ ]:
import glob, shutil
model.save_pretrained_gguf("veronica-gguf", tokenizer, quantization_method="q4_k_m")
gguf = sorted(glob.glob("veronica-gguf/**/*.gguf", recursive=True) + glob.glob("*.gguf"))
print(gguf)
shutil.copy([g for g in gguf if "q4_k_m" in g.lower()][0], "veronica-q4_k_m.gguf")
shutil.copy("gophergeist/finetune/Modelfile", "Modelfile")

from google.colab import files
files.download("veronica-q4_k_m.gguf")
files.download("Modelfile")

## 6. Run it at home

Put `veronica-q4_k_m.gguf` and `Modelfile` in the same folder, then:

```bash
ollama create llama3.2-veronica:1b -f Modelfile
gophergeist doctor --model llama3.2-veronica:1b
gophergeist serve demo/hole --model llama3.2-veronica:1b --web
```

Compare it against the stock model with `gophergeist ask demo/hole "..." --model llama3.2:1b`.
The Llama 3.2 license asks that models built from it carry "Llama" at the start of their name, hence
`llama3.2-veronica`.